In [1]:
# =================================================================================================
# NHANES 2005-2006 DIETARY MERGE
#
# Purpose:
#   Merge DR1TOT_D and DR2TOT_D into one participant-level Dietary CSV.
#
# Final structure:
#
#   SEQN
#   WTDRD1
#   WTDR2D
#   DRABF
#   DRDINT
#   [Day 1 variables]
#   [Day 2 variables]
#
# Fidelity rules:
#   - One SEQN per participant
#   - Preserve Dietary decimals AS IS
#   - Do not round values
#   - Validate shared variables before removing duplicate copies
#   - Keep only one canonical copy of shared variables
#   - Prevent many-to-many merge errors
#   - Day 1 columns appear before Day 2 columns
#
# Source files:
#   DR1TOT_D_corrected.csv
#   DR2TOT_D_corrected.csv
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Dietary"
)


DAY1_FILE = (
    DATA_DIR
    / "DR1TOT_D_corrected.csv"
)


DAY2_FILE = (
    DATA_DIR
    / "DR2TOT_D_corrected.csv"
)


OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed_v3"
    / "2005_2006"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / "neighbourhood_nhanes_dietary_2005_2006_raw_corrected.csv"
)


# =================================================================================================
# 3. READ CORRECTED FILES AS TEXT
# =================================================================================================

#
# IMPORTANT:
#
# dtype=str prevents pandas from changing the corrected
# Dietary decimal representation during this MERGE step.
#
# We are not doing calculations here.
#
# We are only combining the already-corrected CSV files.
#

day1 = pd.read_csv(
    DAY1_FILE,
    dtype=str,
    keep_default_na=False
)


day2 = pd.read_csv(
    DAY2_FILE,
    dtype=str,
    keep_default_na=False
)


print("=" * 100)
print("NHANES 2005-2006 DIETARY DAY 1 + DAY 2 MERGE")
print("=" * 100)

print(
    "DR1TOT_D shape:",
    day1.shape
)

print(
    "DR2TOT_D shape:",
    day2.shape
)


# =================================================================================================
# 4. STANDARDIZE COLUMN NAMES
# =================================================================================================

day1.columns = (
    day1.columns
    .str.strip()
    .str.upper()
)


day2.columns = (
    day2.columns
    .str.strip()
    .str.upper()
)


# =================================================================================================
# 5. VERIFY SEQN EXISTS
# =================================================================================================

for name, df in [
    ("DR1TOT_D", day1),
    ("DR2TOT_D", day2)
]:

    if "SEQN" not in df.columns:

        raise ValueError(
            f"{name} does not contain SEQN."
        )


print(
    "PASS: SEQN exists in both files."
)


# =================================================================================================
# 6. VERIFY EXPECTED ROW COUNTS
# =================================================================================================

#
# CDC 2005-2006 total nutrient files:
#
# DR1TOT_D = 9,950 rows
# DR2TOT_D = 9,950 rows
#

EXPECTED_ROWS = 9950


if len(day1) != EXPECTED_ROWS:

    raise ValueError(
        f"Unexpected DR1TOT_D row count: {len(day1):,}"
    )


if len(day2) != EXPECTED_ROWS:

    raise ValueError(
        f"Unexpected DR2TOT_D row count: {len(day2):,}"
    )


print(
    "PASS: Both files contain 9,950 participant rows."
)


# =================================================================================================
# 7. SEQN FORMAT CHECK
# =================================================================================================

for name, df in [
    ("DR1TOT_D", day1),
    ("DR2TOT_D", day2)
]:

    blank_seqn = int(
        df["SEQN"]
        .eq("")
        .sum()
    )


    dot_zero_seqn = int(
        (
            df["SEQN"].ne("")
            &
            df["SEQN"].str.endswith(".0")
        )
        .sum()
    )


    non_digit_seqn = int(
        (
            df["SEQN"].ne("")
            &
            ~df["SEQN"]
            .str.fullmatch(r"\d+")
        )
        .sum()
    )


    duplicate_seqn = int(
        df["SEQN"]
        .duplicated()
        .sum()
    )


    print(
        f"\n{name}:"
    )

    print(
        "  Blank SEQN:",
        blank_seqn
    )

    print(
        "  SEQN ending .0:",
        dot_zero_seqn
    )

    print(
        "  Non-digit SEQN:",
        non_digit_seqn
    )

    print(
        "  Duplicate SEQN:",
        duplicate_seqn
    )


    if blank_seqn != 0:

        raise ValueError(
            f"{name}: blank SEQN found."
        )


    if dot_zero_seqn != 0:

        raise ValueError(
            f"{name}: SEQN still contains '.0'."
        )


    if non_digit_seqn != 0:

        raise ValueError(
            f"{name}: invalid SEQN formatting."
        )


    if duplicate_seqn != 0:

        raise ValueError(
            f"{name}: duplicate SEQN found."
        )


print(
    "\nPASS: SEQN integrity validated in both files."
)


# =================================================================================================
# 8. VERIFY BOTH FILES HAVE SAME PARTICIPANT UNIVERSE
# =================================================================================================

day1_seqn = set(
    day1["SEQN"]
)


day2_seqn = set(
    day2["SEQN"]
)


only_day1 = sorted(
    day1_seqn
    -
    day2_seqn
)


only_day2 = sorted(
    day2_seqn
    -
    day1_seqn
)


print(
    "\n--- PARTICIPANT SET COMPARISON ---"
)

print(
    "Only in DR1TOT_D:",
    len(only_day1)
)

print(
    "Only in DR2TOT_D:",
    len(only_day2)
)


if only_day1 or only_day2:

    raise ValueError(
        "DR1TOT_D and DR2TOT_D do not contain "
        "the same SEQN universe.\n\n"
        f"Only Day 1: {len(only_day1):,}\n"
        f"Only Day 2: {len(only_day2):,}"
    )


print(
    "PASS: Day 1 and Day 2 contain identical participant sets."
)


# =================================================================================================
# 9. FIND COLUMN NAMES PRESENT IN BOTH FILES
# =================================================================================================

shared_columns_detected = [
    col
    for col in day1.columns
    if (
        col in day2.columns
        and
        col != "SEQN"
    )
]


print(
    "\n--- SHARED VARIABLES DETECTED ---"
)

print(
    shared_columns_detected
)


# =================================================================================================
# 10. EXPECTED SHARED NHANES VARIABLES
# =================================================================================================

#
# CDC explicitly identifies these as the same variables
# in both DR1TOT_D and DR2TOT_D.
#

SHARED_COLUMNS = [
    "WTDRD1",
    "WTDR2D",
    "DRABF",
    "DRDINT"
]


if set(
    shared_columns_detected
) != set(
    SHARED_COLUMNS
):

    raise ValueError(
        "Unexpected shared-column pattern.\n\n"
        f"Expected:\n{SHARED_COLUMNS}\n\n"
        f"Detected:\n{shared_columns_detected}"
    )


print(
    "PASS: Exactly four expected shared NHANES variables found."
)


# =================================================================================================
# 11. COMPARE SHARED VARIABLES BY SEQN
# =================================================================================================

#
# Never simply drop duplicated variables.
#
# First prove that their values are identical.
#

shared_comparison = (
    day1[
        ["SEQN"] + SHARED_COLUMNS
    ]
    .merge(
        day2[
            ["SEQN"] + SHARED_COLUMNS
        ],
        on="SEQN",
        how="outer",
        suffixes=("_DAY1", "_DAY2"),
        validate="one_to_one"
    )
)


print(
    "\n--- SHARED VARIABLE FIDELITY CHECK ---"
)


for col in SHARED_COLUMNS:

    day1_values = (
        shared_comparison[
            f"{col}_DAY1"
        ]
    )


    day2_values = (
        shared_comparison[
            f"{col}_DAY2"
        ]
    )


    mismatch_mask = (
        day1_values
        !=
        day2_values
    )


    mismatch_count = int(
        mismatch_mask.sum()
    )


    print(
        f"{col}: {mismatch_count:,} mismatches"
    )


    if mismatch_count != 0:

        print(
            "\nExample mismatches:"
        )


        print(
            shared_comparison.loc[
                mismatch_mask,
                [
                    "SEQN",
                    f"{col}_DAY1",
                    f"{col}_DAY2"
                ]
            ]
            .head(20)
        )


        raise ValueError(
            f"{col} differs between "
            "DR1TOT_D and DR2TOT_D."
        )


print(
    "\nPASS: WTDRD1, WTDR2D, DRABF and DRDINT "
    "match exactly between Day 1 and Day 2."
)


# =================================================================================================
# 12. SAVE ORIGINAL DAY 1 / DAY 2 COLUMN ORDERS
# =================================================================================================

#
# We capture the original orders before removing anything.
#

original_day1_columns = (
    day1.columns.tolist()
)


original_day2_columns = (
    day2.columns.tolist()
)


# =================================================================================================
# 13. REMOVE ONLY THE REDUNDANT DAY 2 COPIES
# =================================================================================================

#
# Keep:
#
#   WTDRD1
#   WTDR2D
#   DRABF
#   DRDINT
#
# from Day 1.
#
# Remove the IDENTICAL copies from Day 2 before merging.
#
# This prevents pandas from generating:
#
#   WTDRD1_DAY_2_TOTAL_INTAKE
#   WTDR2D_DAY_2_TOTAL_INTAKE
#   DRABF_DAY_2_TOTAL_INTAKE
#   DRDINT_DAY_2_TOTAL_INTAKE
#

day2_for_merge = (
    day2.drop(
        columns=SHARED_COLUMNS
    )
)


print(
    "\nRemoved redundant Day 2 copies:"
)


for col in SHARED_COLUMNS:

    print(
        " -",
        col
    )


# =================================================================================================
# 14. MERGE DAY 1 + DAY 2
# =================================================================================================

#
# validate="one_to_one" is critical.
#
# If either file unexpectedly contains duplicate SEQN,
# pandas will stop rather than silently multiplying rows.
#

dietary = day1.merge(
    day2_for_merge,
    on="SEQN",
    how="outer",
    validate="one_to_one"
)


print(
    "\nPASS: DR1TOT_D + DR2TOT_D merge completed."
)


# =================================================================================================
# 15. VALIDATE MERGED PARTICIPANT COUNT
# =================================================================================================

if len(dietary) != EXPECTED_ROWS:

    raise ValueError(
        "Merged Dietary row count changed.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(dietary):,}"
    )


if dietary["SEQN"].nunique() != EXPECTED_ROWS:

    raise ValueError(
        "Merged Dietary unique SEQN count changed."
    )


if dietary["SEQN"].duplicated().any():

    raise ValueError(
        "Merged Dietary dataset contains duplicate SEQN."
    )


if dietary["SEQN"].str.endswith(".0").any():

    raise ValueError(
        "Merged Dietary dataset contains SEQN '.0'."
    )


print(
    "PASS: Final Dietary file has "
    "9,950 unique participants."
)


# =================================================================================================
# 16. VERIFY NO DUPLICATE COLUMN NAMES
# =================================================================================================

duplicate_column_names = (
    dietary.columns[
        dietary.columns.duplicated()
    ]
    .tolist()
)


if duplicate_column_names:

    raise ValueError(
        "Duplicate column names found:\n"
        + str(duplicate_column_names)
    )


print(
    "PASS: No duplicate column names."
)


# =================================================================================================
# 17. VERIFY ENGINEERED DUPLICATE COLUMNS DO NOT EXIST
# =================================================================================================

BAD_DUPLICATE_COLUMNS = [
    "WTDRD1_DAY_2_TOTAL_INTAKE",
    "WTDR2D_DAY_2_TOTAL_INTAKE",
    "DRABF_DAY_2_TOTAL_INTAKE",
    "DRDINT_DAY_2_TOTAL_INTAKE"
]


existing_bad_columns = [
    col
    for col in BAD_DUPLICATE_COLUMNS
    if col in dietary.columns
]


if existing_bad_columns:

    raise ValueError(
        "Redundant engineered Dietary fields remain:\n"
        + str(existing_bad_columns)
    )


print(
    "PASS: No _DAY_2_TOTAL_INTAKE duplicate columns."
)


# =================================================================================================
# 18. CREATE FINAL LOGICAL COLUMN ORDER
# =================================================================================================

#
# Desired structure:
#
# SEQN
#
# shared variables
#
# DAY 1
#
# DAY 2
#

SHARED_ORDER = [
    "SEQN",
    "WTDRD1",
    "WTDR2D",
    "DRABF",
    "DRDINT"
]


#
# Everything unique to Day 1, preserving original
# DR1TOT_D source order.
#

DAY1_UNIQUE_COLUMNS = [
    col
    for col in original_day1_columns
    if col not in SHARED_ORDER
]


#
# Everything unique to Day 2, preserving original
# DR2TOT_D source order.
#

DAY2_UNIQUE_COLUMNS = [
    col
    for col in original_day2_columns
    if (
        col != "SEQN"
        and
        col not in SHARED_COLUMNS
    )
]


FINAL_COLUMN_ORDER = (
    SHARED_ORDER
    +
    DAY1_UNIQUE_COLUMNS
    +
    DAY2_UNIQUE_COLUMNS
)


# =================================================================================================
# 19. VERIFY FINAL ORDER HAS NO DUPLICATES
# =================================================================================================

if (
    len(FINAL_COLUMN_ORDER)
    !=
    len(set(FINAL_COLUMN_ORDER))
):

    raise ValueError(
        "FINAL_COLUMN_ORDER contains duplicate column names."
    )


if set(
    FINAL_COLUMN_ORDER
) != set(
    dietary.columns
):

    missing = (
        set(dietary.columns)
        -
        set(FINAL_COLUMN_ORDER)
    )

    extra = (
        set(FINAL_COLUMN_ORDER)
        -
        set(dietary.columns)
    )


    raise ValueError(
        "Final column-order mismatch.\n\n"
        f"Missing:\n{sorted(missing)}\n\n"
        f"Extra:\n{sorted(extra)}"
    )


dietary = dietary[
    FINAL_COLUMN_ORDER
]


print(
    "PASS: Final column order created."
)


# =================================================================================================
# 20. DISPLAY FINAL SECTION COUNTS
# =================================================================================================

print(
    "\n--- FINAL STRUCTURE ---"
)

print(
    "Shared variables:",
    len(SHARED_ORDER)
)

print(
    "Day 1 unique variables:",
    len(DAY1_UNIQUE_COLUMNS)
)

print(
    "Day 2 unique variables:",
    len(DAY2_UNIQUE_COLUMNS)
)

print(
    "Total columns:",
    len(dietary.columns)
)


# =================================================================================================
# 21. VERIFY DAY 1 VALUES SURVIVED EXACTLY
# =================================================================================================

#
# Compare the merged dataframe back to the original
# corrected Day 1 input using TEXT equality.
#

print(
    "\n--- DAY 1 POST-MERGE FIDELITY ---"
)


day1_check = (
    day1.merge(
        dietary[
            original_day1_columns
        ],
        on="SEQN",
        how="inner",
        suffixes=("_SOURCE", "_MERGED"),
        validate="one_to_one"
    )
)


day1_failures = []


for col in original_day1_columns:

    if col == "SEQN":

        continue


    source_col = (
        day1_check[
            f"{col}_SOURCE"
        ]
    )


    merged_col = (
        day1_check[
            f"{col}_MERGED"
        ]
    )


    mismatch_count = int(
        (
            source_col
            !=
            merged_col
        )
        .sum()
    )


    if mismatch_count > 0:

        day1_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day1_failures:

    raise ValueError(
        "Day 1 values changed during merge:\n"
        + str(day1_failures)
    )


print(
    "PASS: Every DR1TOT_D value preserved exactly."
)


# =================================================================================================
# 22. VERIFY DAY 2 UNIQUE VALUES SURVIVED EXACTLY
# =================================================================================================

print(
    "\n--- DAY 2 POST-MERGE FIDELITY ---"
)


day2_check_columns = (
    ["SEQN"]
    +
    DAY2_UNIQUE_COLUMNS
)


day2_check = (
    day2[
        day2_check_columns
    ]
    .merge(
        dietary[
            day2_check_columns
        ],
        on="SEQN",
        how="inner",
        suffixes=("_SOURCE", "_MERGED"),
        validate="one_to_one"
    )
)


day2_failures = []


for col in DAY2_UNIQUE_COLUMNS:

    source_col = (
        day2_check[
            f"{col}_SOURCE"
        ]
    )


    merged_col = (
        day2_check[
            f"{col}_MERGED"
        ]
    )


    mismatch_count = int(
        (
            source_col
            !=
            merged_col
        )
        .sum()
    )


    if mismatch_count > 0:

        day2_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day2_failures:

    raise ValueError(
        "Day 2 values changed during merge:\n"
        + str(day2_failures)
    )


print(
    "PASS: Every unique DR2TOT_D value "
    "preserved exactly."
)


# =================================================================================================
# 23. CHECK SHARED VALUES ONE FINAL TIME
# =================================================================================================

#
# The surviving shared values came from Day 1.
#
# Confirm they still equal the original Day 2 values.
#

print(
    "\n--- FINAL SHARED FIELD CHECK ---"
)


final_shared_check = (
    dietary[
        ["SEQN"] + SHARED_COLUMNS
    ]
    .merge(
        day2[
            ["SEQN"] + SHARED_COLUMNS
        ],
        on="SEQN",
        how="inner",
        suffixes=("_FINAL", "_DAY2"),
        validate="one_to_one"
    )
)


for col in SHARED_COLUMNS:

    mismatch_count = int(
        (
            final_shared_check[
                f"{col}_FINAL"
            ]
            !=
            final_shared_check[
                f"{col}_DAY2"
            ]
        )
        .sum()
    )


    print(
        f"{col}: "
        f"{mismatch_count:,} mismatches"
    )


    if mismatch_count != 0:

        raise ValueError(
            f"Final shared field {col} "
            "does not match Day 2 source."
        )


print(
    "PASS: Shared Dietary variables remain "
    "identical to both source files."
)


# =================================================================================================
# 24. DIETARY DECIMAL PROTECTION
# =================================================================================================

#
# No numeric conversion occurred during merge.
#
# We can still inspect representative decimal fields
# to confirm their text contains decimal values.
#

REPRESENTATIVE_DECIMAL_FIELDS = [
    "WTDRD1",
    "WTDR2D",
    "DR1TPROT",
    "DR1TP204",
    "DR2TPROT",
    "DR2TP204"
]


print(
    "\n--- DIETARY DECIMAL PROTECTION ---"
)


for col in REPRESENTATIVE_DECIMAL_FIELDS:

    if col not in dietary.columns:

        continue


    populated = int(
        dietary[col]
        .ne("")
        .sum()
    )


    decimal_tokens = int(
        (
            dietary[col]
            .ne("")
            &
            dietary[col]
            .str.contains(
                ".",
                regex=False
            )
        )
        .sum()
    )


    print(
        f"{col}: "
        f"populated={populated:,}, "
        f"decimal tokens={decimal_tokens:,}"
    )


print(
    "PASS: No Dietary measurement was rounded "
    "or converted during merge."
)


# =================================================================================================
# 25. SAVE FINAL DIETARY FILE
# =================================================================================================

#
# Because the dataframe contains strings, pandas writes the
# corrected CSV tokens without numeric re-parsing.
#

dietary.to_csv(
    OUTPUT_PATH,
    index=False
)


print(
    "\nSaved final Dietary file:"
)

print(
    OUTPUT_PATH
)


# =================================================================================================
# 26. RE-READ FINAL FILE AS TEXT
# =================================================================================================

final_df = pd.read_csv(
    OUTPUT_PATH,
    dtype=str,
    keep_default_na=False
)


# =================================================================================================
# 27. POST-SAVE STRUCTURE CHECK
# =================================================================================================

if final_df.shape != dietary.shape:

    raise ValueError(
        "Saved CSV shape changed."
    )


if (
    final_df.columns.tolist()
    !=
    dietary.columns.tolist()
):

    raise ValueError(
        "Saved CSV column order changed."
    )


if final_df["SEQN"].duplicated().any():

    raise ValueError(
        "Saved CSV contains duplicate SEQN."
    )


if final_df["SEQN"].str.endswith(".0").any():

    raise ValueError(
        "Saved CSV contains SEQN '.0'."
    )


print(
    "PASS: Saved CSV structure and SEQN integrity preserved."
)


# =================================================================================================
# 28. EXACT TEXT FIDELITY AFTER SAVE
# =================================================================================================

#
# This is particularly useful for Dietary decimals.
#
# Every CSV token before and after saving should be identical.
#

post_save_failures = []


for col in dietary.columns:

    same = (
        dietary[col]
        .eq(
            final_df[col]
        )
        .all()
    )


    if not same:

        post_save_failures.append(
            col
        )


if post_save_failures:

    raise ValueError(
        "CSV text fidelity changed during save:\n"
        + str(post_save_failures)
    )


print(
    "PASS: Exact pre-save/post-save text fidelity."
)


# =================================================================================================
# 29. FINAL CHECK FOR ENGINEERED DUPLICATES
# =================================================================================================

final_bad_columns = [
    col
    for col in final_df.columns
    if col in BAD_DUPLICATE_COLUMNS
]


if final_bad_columns:

    raise ValueError(
        "Engineered duplicate columns remain:\n"
        + str(final_bad_columns)
    )


print(
    "PASS: No redundant Day 2 shared-variable copies remain."
)


# =================================================================================================
# 30. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "NHANES 2005-2006 DIETARY MERGE COMPLETE"
)

print(
    "=" * 100
)

print(
    "Rows:",
    f"{len(final_df):,}"
)

print(
    "Unique SEQN:",
    f"{final_df['SEQN'].nunique():,}"
)

print(
    "Columns:",
    f"{len(final_df.columns):,}"
)

print(
    ""
)

print(
    "Final organization:"
)

print(
    "  SEQN"
)

print(
    "  WTDRD1"
)

print(
    "  WTDR2D"
)

print(
    "  DRABF"
)

print(
    "  DRDINT"
)

print(
    "  [Day 1 fields]"
)

print(
    "  [Day 2 fields]"
)

print(
    ""
)

print(
    "Shared field mismatches: 0"
)

print(
    "Duplicate SEQN: 0"
)

print(
    "SEQN .0: 0"
)

print(
    "Engineered duplicate fields: 0"
)

print(
    "Day 1 value changes: 0"
)

print(
    "Day 2 value changes: 0"
)

print(
    "Dietary decimal reformatting: 0"
)

print(
    "=" * 100
)

NHANES 2005-2006 DIETARY DAY 1 + DAY 2 MERGE
DR1TOT_D shape: (9950, 160)
DR2TOT_D shape: (9950, 81)
PASS: SEQN exists in both files.
PASS: Both files contain 9,950 participant rows.

DR1TOT_D:
  Blank SEQN: 0
  SEQN ending .0: 0
  Non-digit SEQN: 0
  Duplicate SEQN: 0

DR2TOT_D:
  Blank SEQN: 0
  SEQN ending .0: 0
  Non-digit SEQN: 0
  Duplicate SEQN: 0

PASS: SEQN integrity validated in both files.

--- PARTICIPANT SET COMPARISON ---
Only in DR1TOT_D: 0
Only in DR2TOT_D: 0
PASS: Day 1 and Day 2 contain identical participant sets.

--- SHARED VARIABLES DETECTED ---
['WTDRD1', 'WTDR2D', 'DRABF', 'DRDINT']
PASS: Exactly four expected shared NHANES variables found.

--- SHARED VARIABLE FIDELITY CHECK ---
WTDRD1: 0 mismatches
WTDR2D: 0 mismatches
DRABF: 0 mismatches
DRDINT: 0 mismatches

PASS: WTDRD1, WTDR2D, DRABF and DRDINT match exactly between Day 1 and Day 2.

Removed redundant Day 2 copies:
 - WTDRD1
 - WTDR2D
 - DRABF
 - DRDINT

PASS: DR1TOT_D + DR2TOT_D merge completed.
PASS: Final